# 07 · Dashboard queries (Week 5 demo)
Each cell is one dashboard tile. In Databricks: run a cell, click **+ → Visualization** on the result,
then **Add to dashboard**. Or paste the queries into a Databricks SQL / AI-BI dashboard dataset.

In [0]:
USE workspace.wildfire;

In [0]:
-- Tile 1 · Statewide trend: fire activity vs average county PM2.5 (line chart, two y-axes)
SELECT date_local, ca_fire_detections, ca_fire_frp_mw, avg_county_pm25_ugm3, counties_over_standard
FROM gold_kpi_state_day ORDER BY date_local;

date_local,ca_fire_detections,ca_fire_frp_mw,avg_county_pm25_ugm3,counties_over_standard
2020-08-15,457,9099.0,9.93,0
2020-08-16,602,16603.0,9.89,0
2020-08-17,1005,30294.0,12.96,0
2020-08-18,2251,71674.0,18.78,3
2020-08-19,10663,386870.0,29.8,16
2020-08-20,9245,179862.0,55.89,24
2020-08-21,5831,88852.0,65.06,25
2020-08-22,3660,57353.0,64.92,27
2020-08-23,4783,104458.0,54.25,26
2020-08-24,2032,64859.0,48.88,21


In [0]:
-- Tile 2 · Worst 15 counties by days over the 24-hour PM2.5 standard (bar chart)
SELECT c.county_name,
       sum(CAST(f.pm25_exceeds_daily_standard AS INT)) AS days_over_35,
       round(max(f.pm25_mean_ugm3), 1) AS worst_day_pm25,
       sum(f.fire_detections) AS fires_in_county
FROM gold_fact_county_day f JOIN gold_dim_county c USING (county_fips)
GROUP BY c.county_name ORDER BY days_over_35 DESC, worst_day_pm25 DESC LIMIT 15;

county_name,days_over_35,worst_day_pm25,fires_in_county
Mono,38,691.4,4372
Fresno,33,158.2,1282
Butte,32,329.3,7863
Tehama,32,142.9,8906
Sutter,30,252.9,34
Kings,30,145.0,56
Colusa,30,119.0,582
Madera,29,199.7,27890
Placer,29,146.0,2
Nevada,27,200.9,2


In [0]:
-- Tile 3 · Smoke indicator mix per study week (stacked bar)
SELECT d.study_week, f.smoke_indicator, count(*) AS county_days
FROM gold_fact_county_day f JOIN gold_dim_date d USING (date_local)
GROUP BY d.study_week, f.smoke_indicator ORDER BY d.study_week;

study_week,smoke_indicator,county_days
1,high_pm25_no_nearby_fire,3
1,clear,28
1,no_pm25_data,113
1,nearby_fire_pm25_ok,197
1,likely_smoke,65
2,high_pm25_no_nearby_fire,4
2,nearby_fire_pm25_ok,153
2,clear,24
2,likely_smoke,110
2,no_pm25_data,115


In [0]:
-- Tile 4 · Does PM2.5 rise with nearby fire activity? (scatter, county-days with PM2.5 data)
SELECT fire_frp_within_100km_mw, pm25_mean_ugm3, wind_mean_ms, county_fips, date_local
FROM gold_fact_county_day WHERE has_pm25;

fire_frp_within_100km_mw,pm25_mean_ugm3,wind_mean_ms,county_fips,date_local
12.0,10.77,2.5,06001,2020-08-15
101.2,18.91,null,06009,2020-08-19
26977.8,37.87,4.6,06013,2020-08-21
0.0,3.92,null,06045,2020-08-15
1184.8,7.32,4.7,06047,2020-08-16
15.6,13.28,3.4,06051,2020-08-18
26865.2,76.27,null,06053,2020-08-19
27719.1,11.23,null,06055,2020-08-20
21.1,5.7,1.6,06093,2020-08-17
29441.2,10.43,2.7,06095,2020-08-18


In [0]:
-- Tile 5 · Lagged view: PM2.5 today vs nearby fire radiative power yesterday (smoke takes time to arrive)
SELECT f.county_fips, f.date_local, f.pm25_mean_ugm3,
       lag(f.fire_frp_within_100km_mw) OVER (PARTITION BY f.county_fips ORDER BY f.date_local) AS frp_100km_prev_day
FROM gold_fact_county_day f WHERE f.has_pm25;

county_fips,date_local,pm25_mean_ugm3,frp_100km_prev_day
06001,2020-08-15,10.77,null
06001,2020-08-16,13.71,12.0
06001,2020-08-17,9.05,2550.9
06001,2020-08-18,13.26,16500.2
06001,2020-08-19,43.5,37471.6
06001,2020-08-20,37.85,179545.5
06001,2020-08-21,44.3,51345.1
06001,2020-08-22,35.51,24282.0
06001,2020-08-23,50.2,17760.6
06001,2020-08-24,34.37,14226.6


In [0]:
-- Tile 6 · Correlation summary for the report (same-day and previous-day)
WITH x AS (
  SELECT pm25_mean_ugm3 AS pm, fire_frp_within_100km_mw AS frp,
         lag(fire_frp_within_100km_mw) OVER (PARTITION BY county_fips ORDER BY date_local) AS frp_lag1, wind_mean_ms
  FROM gold_fact_county_day WHERE has_pm25)
SELECT round(corr(pm, frp), 3) AS corr_same_day, round(corr(pm, frp_lag1), 3) AS corr_prev_day,
       round(corr(pm, wind_mean_ms), 3) AS corr_wind, count(*) AS n_county_days FROM x;

corr_same_day,corr_prev_day,corr_wind,n_county_days
0.139,0.219,-0.113,2656


In [0]:
-- Tile 7 · Data coverage (quality panel)
SELECT date_local, sum(CAST(has_pm25 AS INT)) AS counties_with_pm25, sum(CAST(has_weather AS INT)) AS counties_with_weather,
       count(*) AS counties
FROM gold_fact_county_day GROUP BY date_local ORDER BY date_local;

date_local,counties_with_pm25,counties_with_weather,counties
2020-08-15,41,57,58
2020-08-16,44,57,58
2020-08-17,41,57,58
2020-08-18,40,57,58
2020-08-19,45,57,58
2020-08-20,41,57,58
2020-08-21,41,57,58
2020-08-22,43,57,58
2020-08-23,40,57,58
2020-08-24,40,57,58
